# Best model per metric (interactive)

A streamlined version of the last section of `compare_models.ipynb`. Reads the
results written by `find_best_model.py` (`performance/<gauge>/`), finds the best
model and realization for each selected metric (up to 5), re-runs them and
shows them against the observations in one interactive (plotly) figure:
time series (linear and log scale), flow-duration curve and Q-Q plot.
Hover to read values, click legend entries to hide lines, drag to zoom;
the two time-series panels zoom together.

In [ ]:
import json
import os
import shlex
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import find_best_model as fbm
import marrmot_models as marrmot

warnings.filterwarnings("ignore", category=RuntimeWarning)  # overflow in poor parameter sets

## Settings

In [ ]:
GAUGE = "02249500"     # None = the most recent run in performance/
BEST_BY_METRICS = ["NSE", "KGE", "log_NSE", "pdi_ave", "pdi_low"]  # up to 5: NSE, KGE, log_NSE, pdi_ave,
                       # pdi_ave2, pdi_ave3, pdi_low, pdi_high, PBIAS, KGE_r, KGE_alpha, KGE_beta, correlation,
                       # mean_annual_peak_flow_diff
CONSTRAINTS = {}       # a realization only counts if it meets all of these, e.g.
                       # {"PBIAS": (-15, 15), "NSE": (0, None)}  -> (min, max), None = no bound

## Run the search (optional)

Runs `find_best_model.py` for `GAUGE` with the kernel's own Python, so no
environment activation is needed, and shows its output as it runs. Every required
option is passed here because the script's prompts cannot be answered from a cell. The kernel is busy until the
search ends; for long runs, `run_fbm.ps1` in a terminal is more convenient.

In [ ]:
RUN_SEARCH = False      # True = run find_best_model.py for GAUGE before plotting
N = 200                 # random realizations per model
START_YEAR, END_YEAR = 2015, 2025
N_PROCS = os.cpu_count()
EXTRA_ARGS = "--skip-snow"  # any other options, e.g. '--models "7,11,13" --precip-source nasa'

if RUN_SEARCH:
    if GAUGE is None:
        raise ValueError("Set GAUGE to run the search")
    cmd = [sys.executable, "-u", "find_best_model.py", "--gauge", GAUGE, "--n", str(N),
           "--start-year", str(START_YEAR), "--end-year", str(END_YEAR),
           "--n-procs", str(N_PROCS), *shlex.split(EXTRA_ARGS)]
    # Print each line as it arrives (a ! command on Windows only shows the output at the end)
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            print(line, end="")
    if proc.returncode != 0:
        raise RuntimeError(f"find_best_model.py failed (exit code {proc.returncode})")

## Load the results

In [ ]:
if GAUGE is None:
    runs = sorted(fbm.PERFORMANCE_DIR.glob("*/run_info.json"), key=lambda p: p.stat().st_mtime)
    if not runs:
        raise FileNotFoundError(f"No results in {fbm.PERFORMANCE_DIR}; run find_best_model.py first")
    GAUGE = runs[-1].parent.name

out_dir = fbm.PERFORMANCE_DIR / GAUGE
run_info = json.loads((out_dir / "run_info.json").read_text())

tables = {int(p.name[6:8]): pd.read_parquet(p) for p in sorted(out_dir.glob("model_*.parquet"))}
names = {m: marrmot.MODEL_REGISTRY[m]["name"] for m in tables}

ALL_METRICS = ["NSE", "KGE", "KGE_r", "KGE_alpha", "KGE_beta", "PBIAS", "correlation",
               "mean_annual_peak_flow_diff", "pdi_ave", "pdi_ave2", "pdi_ave3",
               "pdi_low", "pdi_high", "log_NSE"]
# Older runs were scored before some metrics existed: only use the ones every table has
METRICS = [k for k in ALL_METRICS if all(k in t.columns for t in tables.values())]
not_in_run = [k for k in ALL_METRICS if k not in METRICS]


def require_metrics(*keys):
    missing = [k for k in keys if k not in METRICS]
    if missing:
        raise ValueError(f"{missing} not in the results of gauge {GAUGE}: this run was made before "
                         f"they were added; rerun find_best_model.py for it. Available: {METRICS}")

# One row per successful realization of every model, metrics only
# (parameters differ between models, so they stay in `tables`).
results = pd.concat(
    [t.loc[t["error"].isna(), METRICS].reset_index().assign(model=m, name=names[m])
     for m, t in tables.items()],
    ignore_index=True,
)

print(f"Gauge {GAUGE}: {run_info['start_year']}-{run_info['end_year']}, "
      f"{run_info['n_realizations']} realizations per model, ET = {run_info['et_method']}")
print(f"{len(tables)} models with results, {len(results)} successful realizations")
if not_in_run:
    print(f"Metrics not in this run (made before they were added): {', '.join(not_in_run)}")
if run_info["models_removed"]:
    print("Removed (every realization failed):",
          ", ".join(f"{m} {marrmot.MODEL_REGISTRY[int(m)]['name']}" for m in run_info["models_removed"]))

## Ranking and simulation helpers

In [ ]:
TARGETS = {"PBIAS": 0.0, "mean_annual_peak_flow_diff": 0.0, "KGE_alpha": 1.0, "KGE_beta": 1.0}
KEY_METRICS = ["NSE", "KGE", "log_NSE", "pdi_ave", "pdi_low", "PBIAS"]


def goodness(metric, values):
    # Higher is better for every metric: distance to the target for PBIAS/alpha/beta...
    if metric in TARGETS:
        return -(values - TARGETS[metric]).abs()
    return values


def apply_constraints(df, constraints):
    keep = pd.Series(True, index=df.index)
    for col, (lo, hi) in constraints.items():
        if lo is not None:
            keep &= df[col] >= lo
        if hi is not None:
            keep &= df[col] <= hi
    return df[keep]


def rank_models(metric, constraints=None):
    # One row per model: its best realization under `metric` (and that
    # realization's other key metrics), the median over all valid realizations,
    # and how many realizations met the constraints.
    require_metrics(metric, *(constraints or {}))
    df = apply_constraints(results, constraints or {}).dropna(subset=[metric])
    df = df.assign(score=goodness(metric, df[metric]))
    best = df.loc[df.groupby("model")["score"].idxmax()].set_index("model")
    stats = df.groupby("model")[metric].agg(["median", "size"])
    ranking = best.join(stats).sort_values("score", ascending=False)
    ranking.insert(0, "rank", np.arange(1, len(ranking) + 1))
    ranking = ranking.rename(columns={"median": f"median_{metric}", "size": "n_valid"})
    others = [k for k in KEY_METRICS if k != metric]
    return ranking[["rank", "name", "realization", metric, f"median_{metric}", "n_valid"] + others]



def fdc(q):
    q = np.sort(q[np.isfinite(q)])[::-1]
    return 100 * np.arange(1, len(q) + 1) / (len(q) + 1), q

The forcing is rebuilt from `data/` with the same settings as the search
(`run_info.json`).

In [ ]:
forcing = fbm.prepare_data(GAUGE, run_info["start_year"], run_info["end_year"],
                           run_info["warmup_years"], run_info["et_method"],
                           skip_snow=not run_info.get("temperature_available", True),
                           # runs made before AORC support used NASA POWER rain
                           precip_source=run_info.get("precip_source", "nasa"),
                           nasa_sampling=run_info.get("nasa_sampling", "point"))


def simulate_realization(model, realization):
    info = marrmot.MODEL_REGISTRY[model]
    params = tables[model].loc[realization, info["param_names"]].to_dict()
    S0 = np.full(info["n_stores"], fbm.INITIAL_STORAGE)
    result = marrmot.simulate(model, forcing["P"], forcing["Ep"], params, S0=S0, dt=1.0,
                              T=forcing["T"], t=forcing["t"])
    Q = np.maximum(np.asarray(result["Q"], dtype=float)[forcing["eval_mask"]], fbm.Q_FLOOR)
    return pd.DataFrame({"Q": Q, "Qobs": forcing["Qobs"]}, index=forcing["dates"])


## Best model per metric

In [ ]:
GRID, SURFACE, INK = "#e1e0d9", "#fcfcfb", "#3b3b38"
SERIES_5 = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # categorical slots 1-5, fixed order

unknown = [k for k in BEST_BY_METRICS if k not in ALL_METRICS]
if unknown:
    raise ValueError(f"Unknown metric(s) {unknown}; choose from {ALL_METRICS}")
require_metrics(*BEST_BY_METRICS)  # also in this gauge's results?
if not 1 <= len(BEST_BY_METRICS) <= 5:
    raise ValueError("Select between 1 and 5 metrics")

# Winner (model + realization) for each metric
winners = []
for metric in BEST_BY_METRICS:
    r = rank_models(metric, CONSTRAINTS)
    if r.empty:
        raise ValueError(f"No realization satisfies the constraints for {metric}")
    winners.append({"metric": metric, "model": int(r.index[0]), "name": r.iloc[0]["name"],
                    "realization": int(r.iloc[0]["realization"]), "value": r.iloc[0][metric]})
winners = pd.DataFrame(winners)

display(pd.concat(
    [winners.set_index("metric")[["model", "name", "realization"]],
     pd.DataFrame([tables[w.model].loc[w.realization, KEY_METRICS] for w in winners.itertuples()],
                  index=winners["metric"])],
    axis=1,
).style.format(precision=3))

# One line per distinct (model, realization); colors follow metric order
lines = []
for (m, real), grp in winners.groupby(["model", "realization"], sort=False):
    won = ", ".join(f"{k} {v:.3f}" for k, v in zip(grp["metric"], grp["value"]))
    lines.append({"model": m, "realization": real, "color": SERIES_5[len(lines)],
                  "label": f"{m} {grp['name'].iloc[0][:28]} - best {won}",
                  "sim": simulate_realization(m, real)["Q"]})

obs_series = pd.Series(forcing["Qobs"], index=forcing["dates"])
probs = np.linspace(0.01, 0.99, 99)
q_obs = np.quantile(obs_series[obs_series > 0].dropna(), probs)

fig = make_subplots(
    rows=3, cols=2,
    specs=[[{"colspan": 2}, None], [{"colspan": 2}, None], [{}, {}]],
    row_heights=[0.31, 0.31, 0.38], vertical_spacing=0.07, horizontal_spacing=0.08,
    subplot_titles=("Time series - linear scale", "Time series - log scale",
                    "Flow-duration curve", "Q-Q plot"),
)


def add_line(x, y, label, color, group, show_legend, row, col, hover, width=1.4, mode="lines"):
    fig.add_trace(go.Scatter(
        x=x, y=y, name=label, legendgroup=group, showlegend=show_legend, mode=mode,
        line=dict(color=color, width=width), marker=dict(size=5),
        hovertemplate=hover + f"<extra>{label.split(' - ')[0]}</extra>",
    ), row=row, col=col)


series = [{"label": "Observed", "color": "black", "q": obs_series, "group": "obs"}] + \
         [{"label": l["label"], "color": l["color"], "q": l["sim"], "group": f"{l['model']}-{l['realization']}"}
          for l in lines]

for s in series:
    q = s["q"]
    add_line(q.index, q, s["label"], s["color"], s["group"], True, 1, 1, "%{y:.3f} mm/day")
    add_line(q.index, q, s["label"], s["color"], s["group"], False, 2, 1, "%{y:.3f} mm/day")
    exceed, q_sorted = fdc(q.to_numpy())
    add_line(exceed, q_sorted, s["label"], s["color"], s["group"], False, 3, 1,
             "%{y:.3f} mm/day at %{x:.1f}% exceedance")
    if s["group"] != "obs":
        add_line(q_obs, np.quantile(q, probs), s["label"], s["color"], s["group"], False, 3, 2,
                 "obs %{x:.3f} → sim %{y:.3f} mm/day", mode="lines+markers", width=1.2)

lims = [min(q_obs.min(), *(l["sim"].min() for l in lines)),
        max(q_obs.max(), *(l["sim"].max() for l in lines))]
fig.add_trace(go.Scatter(x=lims, y=lims, mode="lines", name="1:1", showlegend=False,
                         line=dict(color=INK, width=1, dash="dash"), hoverinfo="skip"), row=3, col=2)

fig.update_xaxes(matches="x", row=2, col=1)  # the two time-series panels zoom together
fig.update_yaxes(title_text="Flow (mm/day)", row=1, col=1)
fig.update_yaxes(title_text="Flow (mm/day)", type="log", row=2, col=1)
fig.update_yaxes(title_text="Flow (mm/day)", type="log", row=3, col=1)
fig.update_xaxes(title_text="Exceedance probability (%)", range=[0, 100], row=3, col=1)
fig.update_xaxes(title_text="Observed flow (mm/day)", type="log", row=3, col=2)
fig.update_yaxes(title_text="Simulated flow (mm/day)", type="log", row=3, col=2)
fig.update_xaxes(showgrid=True, gridcolor=GRID, linecolor="#8a8984", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor=GRID, linecolor="#8a8984", zeroline=False)
fig.update_layout(
    title=dict(text=f"Gauge {GAUGE}: best model per metric"
                    + (f" (constraints {CONSTRAINTS})" if CONSTRAINTS else ""), x=0.01, y=0.99, yanchor="top"),
    height=1050, template="simple_white", plot_bgcolor=SURFACE, paper_bgcolor=SURFACE,
    hovermode="x unified", margin=dict(t=140, l=70, r=30, b=60),
    legend=dict(orientation="h", x=0, y=1.03, xanchor="left", yanchor="bottom", font=dict(size=11)),
)
fig.show()